# Offline light exposure from SedTRAILS trajectories

This notebook reads saved SedTRAILS particle trajectories, samples Eulerian fields from a SedTRAILS-format input NetCDF, and computes first-pass light exposure diagnostics. The reusable logic lives in `sedtrails.simulation_analysis` so the notebook stays thin.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sedtrails.simulation_analysis import (
    attenuation_from_storlazzi,
    combine_attenuation_components,
    compute_light_exposure,
    sample_field_at_trajectories,
    surface_light_series,
)

## Inputs

`sedtrails_results.nc` provides the saved particle paths. `inlet_sedtrails.nc` provides water depth and suspended sediment concentration on the model grid.

In [ ]:
trajectory_path = Path("examples/results/sedtrails_results.nc")
field_path = Path("sample-data/inlet_sedtrails.nc")

tracks = xr.open_dataset(trajectory_path)
fields = xr.open_dataset(field_path, decode_timedelta=True)

tracks, fields[["waterdepth", "suspended_sed_conc"]]

## Sample depth and concentration along paths

The sampler performs linear interpolation in time and triangular linear interpolation in space. `nearest_fallback=True` keeps this exploratory workflow robust for positions just outside the triangulation; for final analyses, inspect those fallback cases explicitly.

In [ ]:
track_time = tracks["time"].values
track_x = tracks["x"].values
track_y = tracks["y"].values

grid_time = fields["time"].values
grid_x = fields["net_xcc"].values
grid_y = fields["net_ycc"].values

sampled_depth = sample_field_at_trajectories(
    grid_time,
    grid_x,
    grid_y,
    fields["waterdepth"].values,
    track_time,
    track_x,
    track_y,
    nearest_fallback=True,
)

sampled_conc = sample_field_at_trajectories(
    grid_time,
    grid_x,
    grid_y,
    fields["suspended_sed_conc"].values,
    track_time,
    track_x,
    track_y,
    sediment_fraction=0,
    nearest_fallback=True,
)

water_depth_at_particle = sampled_depth.values
depth_avg_concentration = sampled_conc.values

print("sampled water depth range", np.nanmin(water_depth_at_particle), np.nanmax(water_depth_at_particle))
print("sampled concentration range", np.nanmin(depth_avg_concentration), np.nanmax(depth_avg_concentration))

In [ ]:
# First-pass assumptions:
# - use the available concentration field for both sand-only and mud-only sensitivity runs
# - use a near-bed particle depth until the saved trajectory z convention is confirmed
# - zero light when buried; full light when beached; no new exposure outside the domain
components = attenuation_from_storlazzi(
    sed_conc_mud=depth_avg_concentration,
    sed_conc_sand=depth_avg_concentration,
)
kd_sand = combine_attenuation_components(components["mud"], components["sand"], strategy="sand")
kd_mud = combine_attenuation_components(components["mud"], components["sand"], strategy="mud")

track_datetime = track_time.astype("datetime64[ns]")
hour_of_day = (track_datetime.astype("datetime64[h]").astype(int) % 24).astype(float)
daylight = (hour_of_day >= 6.0) & (hour_of_day <= 18.0)
surface_light = surface_light_series(track_datetime, constant=1.0, daylight_mask=daylight)

burial_depth = tracks["burial_depth"].values if "burial_depth" in tracks else None
in_domain = tracks["status_domain"].values.astype(bool) if "status_domain" in tracks else None
is_beached = tracks["status_beached"].values.astype(bool) if "status_beached" in tracks else None

particle_depth = water_depth_at_particle

sand_result = compute_light_exposure(
    track_datetime,
    water_depth_at_particle,
    kd_sand,
    z=particle_depth,
    surface_light=surface_light,
    burial_depth=burial_depth,
    burial_depth_threshold=1e-6,
    in_domain=in_domain,
    is_beached=is_beached,
    light_threshold=0.01,
)

mud_result = compute_light_exposure(
    track_datetime,
    water_depth_at_particle,
    kd_mud,
    z=particle_depth,
    surface_light=surface_light,
    burial_depth=burial_depth,
    burial_depth_threshold=1e-6,
    in_domain=in_domain,
    is_beached=is_beached,
    light_threshold=0.01,
)

In [ ]:
trajectory_id = tracks["trajectory_id"].values if "trajectory_id" in tracks else np.arange(track_x.shape[1])
summary = pd.DataFrame({
    "trajectory_id": trajectory_id,
    "sand_cumulative_light_dose": sand_result.cumulative_light_dose[-1],
    "sand_equivalent_sunlight_hours": sand_result.equivalent_sunlight_hours[-1],
    "sand_time_above_threshold_s": sand_result.time_above_threshold[-1],
    "mud_cumulative_light_dose": mud_result.cumulative_light_dose[-1],
    "mud_equivalent_sunlight_hours": mud_result.equivalent_sunlight_hours[-1],
    "mud_time_above_threshold_s": mud_result.time_above_threshold[-1],
    "mean_sampled_depth_m": np.nanmean(water_depth_at_particle, axis=0),
    "mean_sampled_concentration_kg_m3": np.nanmean(depth_avg_concentration, axis=0),
})
summary

In [ ]:
time_hours = (track_datetime - track_datetime[0]).astype("timedelta64[s]").astype(float) / 3600.0

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for particle_index in range(track_x.shape[1]):
    label = f"particle {trajectory_id[particle_index]}"
    axes[0].plot(time_hours, water_depth_at_particle[:, particle_index], label=label)
    axes[1].plot(time_hours, depth_avg_concentration[:, particle_index], label=label)
    axes[2].plot(time_hours, sand_result.cumulative_light_dose[:, particle_index], label=label)

axes[0].set_ylabel("water depth [m]")
axes[1].set_ylabel("SSC [kg/m3]")
axes[2].set_ylabel("cum. light dose")
axes[2].set_xlabel("time since release [h]")
for ax in axes:
    ax.grid(True, alpha=0.3)
axes[0].legend(loc="best")

## Particle depth and burial

The shaded intervals show timesteps where burial depth exceeds the light-exposure burial threshold.

In [ ]:
burial_threshold = 1e-6
burial_binary = np.zeros_like(particle_depth, dtype=bool) if burial_depth is None else burial_depth > burial_threshold

fig, axes = plt.subplots(track_x.shape[1], 1, figsize=(9, 2.8 * track_x.shape[1]), sharex=True)
axes = np.atleast_1d(axes)

for particle_index, ax in enumerate(axes):
    ax.plot(
        time_hours,
        particle_depth[:, particle_index],
        color="tab:blue",
        lw=1.8,
        label="particle depth below surface",
    )
    ax.plot(
        time_hours,
        water_depth_at_particle[:, particle_index],
        color="0.45",
        lw=1.0,
        ls="--",
        label="sampled water depth",
    )

    buried = burial_binary[:, particle_index]
    starts = np.flatnonzero(np.diff(np.r_[False, buried]) == 1)
    stops = np.flatnonzero(np.diff(np.r_[buried, False]) == -1)
    for start, stop in zip(starts, stops, strict=True):
        right = time_hours[max(stop - 1, start)]
        if right == time_hours[start] and start + 1 < time_hours.size:
            right = time_hours[start + 1]
        ax.axvspan(time_hours[start], right, color="tab:orange", alpha=0.18, lw=0)

    ax.set_ylabel("depth [m]")
    ax.set_title(f"particle {trajectory_id[particle_index]}")
    ax.grid(True, alpha=0.3)
    ax.invert_yaxis()

axes[-1].set_xlabel("time since release [h]")
axes[0].legend(loc="best")

## Next steps

- Confirm the vertical convention for saved trajectory `z` and replace the near-bed approximation.
- Split sand and mud concentration inputs when both fields are available.
- Save the sampled path fields and exposure diagnostics to NetCDF once the variable names stabilize.
- Replace placeholder bleaching and OSL signal arrays with colleague-provided equations.